# 02 — Data Cleaning & Validation
**Bluestock Fintech MF Capstone**

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Ensure CWD is project root (one level up from notebooks/)
PROJECT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

RAW = Path("data/raw")
PROC = Path("data/processed")

nav = pd.read_csv(RAW / "02_nav_history.csv")
print("\n=== Raw NAV History ===")
print(f"Shape: {nav.shape}")
print(f"\nNull counts:\n{nav.isnull().sum()}")
print(f"\nDtypes:\n{nav.dtypes}")

Working directory: C:\Users\prati\.gemini\antigravity\scratch\bluestock_project

=== Raw NAV History ===
Shape: (46000, 3)

Null counts:
amfi_code    0
date         0
nav          0
dtype: int64

Dtypes:
amfi_code      int64
date             str
nav          float64
dtype: object


In [2]:
shape_before = nav.shape
print(f"Shape BEFORE cleaning: {shape_before}")

# Parse dates
nav["date"] = pd.to_datetime(nav["date"], errors="coerce")

# Sort by amfi_code and date
nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# Forward-fill NAV within each scheme
nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()

# Remove duplicate (amfi_code, date) rows
nav = nav.drop_duplicates(subset=["amfi_code", "date"], keep="last")

# Validate nav > 0
invalid_nav = (nav["nav"] <= 0).sum()
print(f"Rows with nav <= 0: {invalid_nav}")
nav = nav[nav["nav"] > 0].reset_index(drop=True)

shape_after = nav.shape
print(f"Shape AFTER  cleaning: {shape_after}")
print(f"Rows removed: {shape_before[0] - shape_after[0]}")
print(f"Remaining nulls: {nav.isnull().sum().sum()}")

Shape BEFORE cleaning: (46000, 3)
Rows with nav <= 0: 0
Shape AFTER  cleaning: (46000, 3)
Rows removed: 0
Remaining nulls: 0


In [3]:
txn = pd.read_csv(RAW / "08_investor_transactions.csv")
print("=== Raw Investor Transactions ===")
print(f"Shape: {txn.shape}")
print(f"\ntransaction_type value_counts:\n{txn['transaction_type'].value_counts()}")
print(f"\nNull amounts: {txn['amount_inr'].isnull().sum()}")
print(f"\nTotal nulls per column:\n{txn.isnull().sum()}")

=== Raw Investor Transactions ===
Shape: (32778, 13)

transaction_type value_counts:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Null amounts: 0

Total nulls per column:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64


In [4]:
shape_before_txn = txn.shape
print(f"Shape BEFORE cleaning: {shape_before_txn}")

# Standardize transaction_type
type_map = {
    "sip": "SIP", "Sip": "SIP", "SIP": "SIP",
    "lumpsum": "Lumpsum", "Lumpsum": "Lumpsum", "LUMPSUM": "Lumpsum", "lump_sum": "Lumpsum",
    "redemption": "Redemption", "Redemption": "Redemption", "REDEMPTION": "Redemption",
}
txn["transaction_type"] = txn["transaction_type"].map(
    lambda x: type_map.get(str(x).strip(), str(x).strip())
)
print(f"\ntransaction_type after standardisation:\n{txn['transaction_type'].value_counts()}")

# Validate amount > 0
neg_amounts = (txn["amount_inr"] <= 0).sum()
print(f"\nRows with amount <= 0: {neg_amounts}")
txn = txn[txn["amount_inr"] > 0].reset_index(drop=True)

# Parse dates
txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], errors="coerce")

shape_after_txn = txn.shape
print(f"\nShape AFTER  cleaning: {shape_after_txn}")
print(f"Rows removed: {shape_before_txn[0] - shape_after_txn[0]}")

Shape BEFORE cleaning: (32778, 13)



transaction_type after standardisation:
transaction_type
SIP           19716
Lumpsum        8095
Redemption     4967
Name: count, dtype: int64

Rows with amount <= 0: 0

Shape AFTER  cleaning: (32778, 13)
Rows removed: 0


In [5]:
perf = pd.read_csv(RAW / "07_scheme_performance.csv")
print("=== Scheme Performance Validation ===")
print(f"Shape: {perf.shape}")

# Check numeric columns
num_cols = ["return_1yr_pct","return_3yr_pct","return_5yr_pct","sharpe_ratio",
            "sortino_ratio","std_dev_ann_pct","max_drawdown_pct","aum_crore","expense_ratio_pct"]
for c in num_cols:
    if c in perf.columns:
        perf[c] = pd.to_numeric(perf[c], errors="coerce")

# Flag negative Sharpe
neg_sharpe = perf[perf["sharpe_ratio"] < 0]
print(f"\nSchemes with negative Sharpe ratio: {len(neg_sharpe)}")
if len(neg_sharpe) > 0:
    print(neg_sharpe[["amfi_code","scheme_name","sharpe_ratio"]])

# Validate expense_ratio between 0.1 and 2.5
out_of_range = perf[(perf["expense_ratio_pct"] < 0.1) | (perf["expense_ratio_pct"] > 2.5)]
print(f"\nSchemes with expense_ratio outside [0.1, 2.5]: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range[["amfi_code","scheme_name","expense_ratio_pct"]])

print(f"\nNull counts:\n{perf.isnull().sum()}")
print("\n✅ Scheme performance validated.")

=== Scheme Performance Validation ===
Shape: (40, 19)

Schemes with negative Sharpe ratio: 0

Schemes with expense_ratio outside [0.1, 2.5]: 0

Null counts:
amfi_code             0
scheme_name           0
fund_house            0
category              0
plan                  0
return_1yr_pct        0
return_3yr_pct        0
return_5yr_pct        0
benchmark_3yr_pct     0
alpha                 0
beta                  0
sharpe_ratio          0
sortino_ratio         0
std_dev_ann_pct       0
max_drawdown_pct      0
aum_crore             0
expense_ratio_pct     0
morningstar_rating    0
risk_grade            0
dtype: int64

✅ Scheme performance validated.


In [6]:
# Save cleaned DataFrames to data/processed/
PROC.mkdir(parents=True, exist_ok=True)

nav.to_csv(PROC / "02_nav_history.csv", index=False)
print(f"✅ Saved cleaned nav_history → {PROC / '02_nav_history.csv'}  ({nav.shape})")

txn.to_csv(PROC / "08_investor_transactions.csv", index=False)
print(f"✅ Saved cleaned investor_transactions → {PROC / '08_investor_transactions.csv'}  ({txn.shape})")

perf.to_csv(PROC / "07_scheme_performance.csv", index=False)
print(f"✅ Saved cleaned scheme_performance → {PROC / '07_scheme_performance.csv'}  ({perf.shape})")

# For the remaining CSVs, copy as-is if not already present
import shutil
for i in ["01","03","04","05","06","09","10"]:
    raw_files = list(RAW.glob(f"{i}_*.csv"))
    for rf in raw_files:
        dest = PROC / rf.name
        if not dest.exists():
            shutil.copy2(rf, dest)
            print(f"  Copied {rf.name} → processed/")
        else:
            print(f"  {rf.name} already exists in processed/")

✅ Saved cleaned nav_history → data\processed\02_nav_history.csv  ((46000, 3))


✅ Saved cleaned investor_transactions → data\processed\08_investor_transactions.csv  ((32778, 13))
✅ Saved cleaned scheme_performance → data\processed\07_scheme_performance.csv  ((40, 19))
  01_fund_master.csv already exists in processed/
  03_aum_by_fund_house.csv already exists in processed/
  04_monthly_sip_inflows.csv already exists in processed/
  05_category_inflows.csv already exists in processed/
  06_industry_folio_count.csv already exists in processed/
  09_portfolio_holdings.csv already exists in processed/
  10_benchmark_indices.csv already exists in processed/


In [7]:
csv_names = [
    "01_fund_master.csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv",
]

print("=" * 65)
print(f"{'Dataset':<35} {'Raw Rows':>10} {'Clean Rows':>12}")
print("=" * 65)

for name in csv_names:
    raw_path = RAW / name
    proc_path = PROC / name
    raw_rows = len(pd.read_csv(raw_path))
    if proc_path.exists():
        clean_rows = len(pd.read_csv(proc_path))
    else:
        clean_rows = raw_rows
    diff = raw_rows - clean_rows
    flag = f"  (-{diff})" if diff > 0 else ""
    print(f"{name:<35} {raw_rows:>10,} {clean_rows:>12,}{flag}")

print("=" * 65)

Dataset                               Raw Rows   Clean Rows
01_fund_master.csv                          40           40
02_nav_history.csv                      46,000       46,000
03_aum_by_fund_house.csv                    90           90


04_monthly_sip_inflows.csv                  48           48
05_category_inflows.csv                    144          144
06_industry_folio_count.csv                 21           21
07_scheme_performance.csv                   40           40


08_investor_transactions.csv            32,778       32,778
09_portfolio_holdings.csv                  322          322
10_benchmark_indices.csv                 8,050        8,050


## Summary
All datasets cleaned and validated. Files saved to data/processed/.